In [ ]:
import stlrom as stl
from stlrom import RandSignalGen, OscillSignalGen
from matplotlib.ticker import MultipleLocator

In [ ]:
osc_gen = OscillSignalGen()
rand_gen = RandSignalGen(seed=1, interp='LINEAR', dt_min=0.02, dt_max=0.8, v_min=-1, v_max=1)

sin_sig = osc_gen.get_signal(t0=0, tf=5, dt=0.02, damp=-0.5, period=1)
rand_sig = rand_gen.get_signal(t0=0, tf=5)

ax = sin_sig.plot(label='Oscill')
ax = rand_sig.plot(label='Rand', ax=ax)

In [ ]:
driver = stl.STLDriver()
driver.parse_string(
    """
    signal x, y

    x_sup := x[t] > 0
    y_sup := y[t] > 0

    rho_x := rho(x_sup)
    sat_x := sat(x_sup)
    ltr_x := ltr(x_sup)
    rtr_x := rtr(x_sup)

    sat_x_n_y := sat(x_sup and y_sup)
    not_sat_x := not sat(x_sup)
    ltr_x_or_rtr_y := ltr(x_sup) or rtr(y_sup)
    ev_ltr_x := ev_[0, 0.5] ltr(x_sup)
    nested := sat(ltr(x_sup) and y_sup)
    """
)
driver.set_signals([sin_sig, rand_sig])
print(driver)

In [ ]:
def show(phis, t_begin=0, t_end=5, signals=(('x sig', sin_sig),), title=None):
    ax = None
    for name, sig in signals:
        ax = sig.plot(label=name, ax=ax)
    for phi in phis:
        rob = driver.get_rob_signal(phi, t_begin, t_end)
        ax = rob.plot(label=phi, ax=ax, draw_samples=False)
    ax.xaxis.set_minor_locator(MultipleLocator(0.1))
    ax.grid(True, which='minor', axis='x')
    ax.set_title(title or f'get_rob_signal between t_begin={t_begin} and t_end={t_end}')
    return ax

In [ ]:
# Each operator alone (global semantics SPACE): rho = space robustness, sat = 1 / -Eps,
# ltr = time since x last changed sign, rtr = time until x next changes sign
for phi in ['rho_x', 'sat_x', 'ltr_x', 'rtr_x']:
    show([phi])

In [ ]:
# The operators should give the same result as the global semantics
sems = {'rho_x': 'SPACE', 'sat_x': 'BOOLEAN', 'ltr_x': 'LEFT_TIME', 'rtr_x': 'RIGHT_TIME'}
for phi, sem in sems.items():
    rob_op = driver.get_rob_signal(phi, 0, 5)
    driver.set_semantics(sem)
    rob_glob = driver.get_rob_signal('x_sup', 0, 5)
    driver.set_semantics('SPACE')
    ax = rob_glob.plot(label=f'x_sup, global {sem}', linewidth=4, alpha=0.5)
    rob_op.plot(label=phi, ax=ax, title=f'{phi} vs global {sem}')

In [ ]:
# Local operator overrides the global semantics: rho(x_sup) stays SPACE under global BOOLEAN
driver.set_semantics('BOOLEAN')
show(['x_sup', 'rho_x'], title='global BOOLEAN')
driver.set_semantics('SPACE')

In [ ]:
# Time semantics on a sub-window: the atom still looks at all the data,
# so the values should match the full-range plot, cut to the window
for phi in ['ltr_x', 'rtr_x']:
    for t_begin, t_end in [[0, 5], [1.2, 3.7]]:
        show([phi], t_begin, t_end)

In [ ]:
# Combinations with boolean and temporal operators
two_sigs = (('x sig', sin_sig), ('y sig', rand_sig))
show(['sat_x_n_y'], signals=two_sigs)
show(['sat_x', 'not_sat_x'])
show(['ltr_x', 'rtr_x'], signals=())
show(['ltr_x_or_rtr_y'], signals=two_sigs)
show(['ltr_x', 'ev_ltr_x'], signals=())

In [ ]:
# Innermost wins: in sat(ltr(x_sup) and y_sup), x_sup is LEFT_TIME and y_sup is BOOLEAN
show(['nested'], signals=two_sigs)